# Project A - Session 0: compactor precondition

Establishes whether the compactor's keep/drop decision carries any salience signal.
**Nothing downstream is worth running until this passes.**

Pass condition: salience lift beats the positional control printed alongside it, with
zero fallbacks and zero empty keeps.

Expected: only `Qwen2.5-1.5B-Instruct` with `backend: scoring` clears it (2.56x vs a
1.68x control on 6 eval trajectories). Budget ~1 GPU-hour.

In [ ]:
!pip -q install peft accelerate
%cd /kaggle/working
import os
REPO = '/kaggle/working/myrios'
if not os.path.exists(REPO):
    !git clone -q https://github.com/USER/myrios.git {REPO}
%cd {REPO}

In [ ]:
!python scripts/preflight.py --config configs/kaggle.yaml --require-gpu

## Wider eval set

The CPU measurements used 6 eval trajectories (36 fact spans), where the margin over the
positional control rests on about 2.7 sigma. Generate more before trusting it.

In [ ]:
!python data/generate_synthetic.py --n-train 48 --n-eval 24 --n-turns 120

## Scorer hand-check

Cheap sanity pass before the full sweep: do obviously salient lines outscore obvious
filler? A non-positive separation means the sweep is pointless.

In [ ]:
for m in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']:
    !python compactor/inspect_scorer.py --config configs/kaggle.yaml --set model.base={m}

## Salience lift per compactor

Both elicitation formats, so the index-list failure is reproduced on GPU rather than
taken on trust from the CPU runs.

In [ ]:
MODELS = ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
for backend in ['scoring', 'model']:
    for m in MODELS:
        slug = m.split('/')[-1] + '_' + backend
        out = f'/kaggle/working/artifacts/runs/compactor_check/{slug}'
        print('=' * 70)
        print(slug)
        !python baselines/cascading.py --config configs/kaggle.yaml --split eval \
            --out {out} --set model.base={m} compaction.backend={backend}
        !python eval/span_report.py --events {out}/eval_events.jsonl --show 0 \
            --out {out}/span_report.json

In [ ]:
import json, glob
rows = []
for p in sorted(glob.glob('/kaggle/working/artifacts/runs/compactor_check/*/span_report.json')):
    r = json.load(open(p))
    rows.append((p.split('/')[-2], r['fact_keep_rate'], r['filler_keep_rate'],
                 r['salience_lift'], r['positional_control']['salience_lift']))
print(f"{'config':<40} {'fact':>6} {'filler':>7} {'lift':>7} {'control':>8}  verdict")
for name, f, fl, lift, ctrl in rows:
    verdict = 'USABLE' if lift > ctrl * 1.1 else 'no signal'
    print(f'{name:<40} {f:6.3f} {fl:7.3f} {lift:7.2f} {ctrl:8.2f}  {verdict}')

In [ ]:
!python scripts/kaggle_sync.py save --run-root /kaggle/working/artifacts/runs \
    --archive /kaggle/working/runs.zip
print('Save /kaggle/working/runs.zip as a Kaggle Dataset before the session ends.')